# The role of neuromodulators in gain modulation

Neuromodulators are chemicals such as dopamine or noradrenaline that do **not** simply
pass a message from one neuron to the next. Instead, they reconfigure *how* circuits
compute — adjusting excitability, gain, signal-to-noise and the balance between
sensory drive and internal dynamics. This tutorial connects the **receptor
biology** of neuromodulation to a concrete **computational picture** using a minimal
leaky integrate-and-fire (LIF) neuron.

A neurotransmitter or neuromodulator can act through two broad receptor families:

- **Ionotropic (fast) receptors** open an ion channel directly. The effect is fast and
  transient and is well summarised as a **change in the input current** arriving at an
  otherwise unchanged neuron — *what enters the neuron*.
- **Metabotropic (slow) receptors** act through intracellular signalling cascades
  (G-proteins, second messengers) that modify intrinsic properties such as the membrane
  time constant $\tau_m$, the spike threshold $V_{th}$, and adaptation currents. The
  effect is slower and longer-lasting and is best summarised as a **change in the
  neuron's own dynamics** — *how the neuron responds*.

Most neuromodulators act predominantly (though not exclusively) through metabotropic
receptors, which is why they are so well suited to reconfiguring computation rather than
just transmitting signals.

### What we will do

1. Distinguish ionotropic vs metabotropic modulation in an LIF neuron.
2. Survey the biological and *theorised* computational roles of the five major neuromodulators.
3. Use noradrenaline (NA) as a worked example of **gain modulation** — scaling the f–I curve.
4. Connect that gain change back to the **metabotropic levers** (intrinsic parameters) that produce it.


## Part 0 — The five major neuromodulators

The brain uses a small set of diffusely projecting neuromodulatory systems to set the
*operating regime* of large populations of neurons. The table summarises their main
biological role, a widely **theorised** computational role, the corresponding "lever"
you would turn in a model, and the resulting network-level effect.

| Neuromodulator | Key biological role | Theorised computational role | Model "lever" | Network-level effect |
|---|---|---|---|---|
| **Dopamine (DA)** | Reward, motivation, movement; phasic bursts to better-than-expected outcomes | Reward-prediction-error / teaching signal; third factor gating plasticity | Reward term $r$; eligibility-trace / learning-rate gate | Decides *what is learned* by scaling synaptic updates |
| **Noradrenaline (NA)** | Arousal, vigilance; broadcast from locus coeruleus | **Gain / signal-to-noise**; unexpected uncertainty; network reset and explore↔exploit | **Input gain $g$ ( `I_syn_gain` )** acting on the f–I slope | Uniformly rescales responsiveness of a population |
| **Acetylcholine (ACh)** | Attention, wakefulness, memory encoding | Expected uncertainty; SNR; encoding-vs-recall balance; effective learning rate | Effective threshold / adaptation; sensory-vs-recurrent weighting | Boosts feedforward (sensory) drive relative to recurrent drive |
| **Serotonin (5-HT)** | Mood, patience, behavioural inhibition | Average-reward / time-horizon; aversive valence; behavioural inhibition | Inhibition bias; reward time-discount | Slows or withholds action; sets baseline tone |
| **Histamine (HA)** | Wakefulness, sleep–wake switching | Global arousal / state gating (awake vs quiescent) | Global excitability bias | Sets the overall operating regime of the network |

> **Note.** The computational roles are *theories*, not settled facts, and the systems
> overlap and interact. Two influential framings are Doya's *metalearning* view
> (DA ≈ TD error, ACh ≈ learning rate, NA ≈ randomness/gain, 5-HT ≈ reward time-scale)
> and the Yu–Dayan view (ACh ≈ expected uncertainty, NA ≈ unexpected uncertainty).

### 💬 Discussion — biological and computational roles

1. Dopamine is often described as a "teaching signal" while noradrenaline is described as
   a "gain" signal. In your own words, what is the difference between *changing what a
   network learns* and *changing how strongly it responds right now*?
2. NA and ACh have both been linked to "uncertainty". Why might it be useful for the brain
   to separate *expected* uncertainty (ACh) from *unexpected* uncertainty (NA)?
3. The same neuromodulator can act through both ionotropic and metabotropic receptors.
   Which of the computational roles above feel more naturally "fast/ionotropic", and which
   feel more naturally "slow/metabotropic"?


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2.2,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ---- Simulation constants -------------------------------------------------
dt          = 0.05     # ms, integration step
T           = 120.0    # ms, voltage-trace window
t           = np.arange(0.0, T, dt)

E_L         = -70.0    # leak / resting potential (mV)
V_reset     = -75.0    # post-spike reset (mV)
R           = 10.0     # membrane resistance (arbitrary scaling)
pulse_start = 20.0     # ms
V_th0       = -50.0    # baseline spike threshold (mV)
tau0        = 20.0     # baseline membrane time constant (ms)

# ---- Colours --------------------------------------------------------------
GREY   = "#9aa3ad"           # baseline / reference traces
PURPLE = "#6f2dbd"           # ionotropic (fast)
BLUE   = "#2d6cdf"           # metabotropic (slow)
BLUE_L = "#7fb6ff"


def make_rect_pulse(time, t0, width_ms, amp):
    """A rectangular input-current pulse."""
    I = np.zeros_like(time)
    I[(time >= t0) & (time < t0 + width_ms)] = amp
    return I


def simulate_lif(time, I, tau_m, v_th, dt_=dt):
    """Leaky integrate-and-fire. Returns (voltage trace, spike times)."""
    V = np.full_like(time, E_L)
    spikes = []
    for k in range(1, len(time)):
        dv = (-(V[k - 1] - E_L) + R * I[k - 1]) * (dt_ / tau_m)
        V[k] = V[k - 1] + dv
        if V[k] >= v_th:
            V[k - 1] = 20.0          # visual spike marker
            V[k] = V_reset           # reset
            spikes.append(time[k])
    return V, np.array(spikes)


def simulate_alif(time, I, tau_m, v_th, b, tau_w=100.0, dt_=dt):
    """LIF + spike-triggered adaptation current w.

    tau_m dV/dt = -(V - E_L) + R (I - w)
    tau_w dw/dt = -w           ; on each spike  w <- w + b
    `b` is the adaptation strength: large b => strong spike-frequency adaptation.
    """
    V = np.full_like(time, E_L)
    w = np.zeros_like(time)
    spikes = []
    for k in range(1, len(time)):
        dv = (-(V[k - 1] - E_L) + R * (I[k - 1] - w[k - 1])) * (dt_ / tau_m)
        dw = (-w[k - 1]) * (dt_ / tau_w)
        V[k] = V[k - 1] + dv
        w[k] = w[k - 1] + dw
        if V[k] >= v_th:
            V[k - 1] = 20.0
            V[k] = V_reset
            w[k] += b
            spikes.append(time[k])
    return V, w, np.array(spikes)


def firing_rate(amp, tau_m, v_th, gain=1.0, adapt_b=0.0, T_fi=500.0, dt_fi=0.1):
    """Steady-state firing rate (Hz) for a sustained input of amplitude `amp`."""
    tt = np.arange(0.0, T_fi, dt_fi)
    I = np.full_like(tt, gain * amp)
    if adapt_b > 0:
        _, _, spikes = simulate_alif(tt, I, tau_m, v_th, b=adapt_b, dt_=dt_fi)
    else:
        _, spikes = simulate_lif(tt, I, tau_m, v_th, dt_=dt_fi)
    return len(spikes) / (T_fi / 1000.0)


def fi_curve(amps, tau_m, v_th, gain=1.0, adapt_b=0.0):
    """f-I curve: firing rate vs input amplitude."""
    return np.array([firing_rate(a, tau_m, v_th, gain, adapt_b) for a in amps])


## Part 1 — Ionotropic vs metabotropic in an LIF neuron

The LIF neuron integrates input current and fires when it reaches threshold:

$$\tau_m \frac{dV}{dt} = -(V - E_L) + R\,I(t), \qquad V \ge V_{th}\ \Rightarrow\ \text{spike, } V \leftarrow V_{reset}.$$

We compare a fixed **baseline** neuron (grey) against a **modulated** one:

- **Ionotropic (left, purple):** the neuron is unchanged; only the **input pulse**
  is scaled (amplitude × `ion_gain`). This is *what enters the neuron*.
- **Metabotropic (right, blue):** the **same** input is delivered, but two intrinsic
  parameters change — the membrane time constant $\tau_m$ and the threshold $V_{th}$.
  This is *how the neuron responds*.

Use the sliders to reach the same spike count by two different routes.

In [7]:
def compare_modulation(ion_gain=1.6, ion_width=70,
                       tau_m=12.0, v_th=-53.0):
    base_amp = 2.2
    base_I = make_rect_pulse(t, pulse_start, 70, base_amp)
    V_base, s_base = simulate_lif(t, base_I, tau0, V_th0)

    ion_I = make_rect_pulse(t, pulse_start, ion_width, base_amp * ion_gain)
    V_ion, s_ion = simulate_lif(t, ion_I, tau0, V_th0)             # neuron fixed

    V_meta, s_meta = simulate_lif(t, base_I, tau_m, v_th)          # input fixed

    fig, ax = plt.subplots(2, 2, figsize=(12, 6.5),
                           gridspec_kw={"height_ratios": [0.6, 1.6]}, sharex=True)

    ax[0, 0].plot(t, base_I, color=GREY, label="baseline input")
    ax[0, 0].plot(t, ion_I, color=PURPLE, label="modulated input")
    ax[0, 0].set_title("Ionotropic: change the INPUT", color=PURPLE, fontweight="bold")
    ax[0, 0].set_ylabel("input"); ax[0, 0].legend(frameon=False, fontsize=9)

    ax[1, 0].axhline(V_th0, color="gray", ls="--", alpha=0.7)
    ax[1, 0].plot(t, V_base, color=GREY, label=f"baseline: {len(s_base)} spikes")
    ax[1, 0].plot(t, V_ion, color=PURPLE, label=f"modulated: {len(s_ion)} spikes")
    ax[1, 0].set_ylabel("voltage (mV)"); ax[1, 0].set_xlabel("time (ms)")
    ax[1, 0].legend(frameon=False)

    ax[0, 1].plot(t, base_I, color=GREY, label="fixed input")
    ax[0, 1].set_title("Metabotropic: change the NEURON", color=BLUE, fontweight="bold")
    ax[0, 1].set_ylabel("input"); ax[0, 1].legend(frameon=False, fontsize=9)

    ax[1, 1].axhline(V_th0, color="gray", ls="--", alpha=0.7, label="baseline $V_{th}$")
    ax[1, 1].axhline(v_th, color=BLUE, ls="--", alpha=0.7, label="modulated $V_{th}$")
    ax[1, 1].plot(t, V_base, color=GREY, label=f"baseline: {len(s_base)} spikes")
    ax[1, 1].plot(t, V_meta, color=BLUE,
                  label=fr"$\tau_m$={tau_m:.0f} ms: {len(s_meta)} spikes")
    ax[1, 1].set_ylabel("voltage (mV)"); ax[1, 1].set_xlabel("time (ms)")
    ax[1, 1].legend(frameon=False)

    for a in ax.ravel():
        a.set_xlim(0, T)
    fig.tight_layout(); plt.show()


interact(
    compare_modulation,
    ion_gain=FloatSlider(value=1.6, min=0.5, max=3.0, step=0.1,
                         description="ion gain", continuous_update=False),
    ion_width=IntSlider(value=70, min=10, max=100, step=5,
                        description="ion width", continuous_update=False),
    tau_m=FloatSlider(value=12.0, min=5.0, max=60.0, step=1.0,
                      description="meta \u03c4_m", continuous_update=False),
    v_th=FloatSlider(value=-53.0, min=-60.0, max=-42.0, step=1.0,
                     description="meta V_th", continuous_update=False),
);


interactive(children=(FloatSlider(value=1.6, continuous_update=False, description='ion gain', max=3.0, min=0.5…

**Interpretation.** On the left the membrane equation never changes — the different
behaviour comes entirely from a stronger or longer input pulse. This is the natural way to
represent **fast, ionotropic** effects. On the right the input is held fixed but the
neuron's *dynamical regime* changes: a smaller $\tau_m$ integrates faster but also leaks
faster, and a lower $V_{th}$ makes spikes easier to generate. This is the natural way to
represent **slow, metabotropic** effects. The key idea for the rest of the tutorial:
neuromodulators mostly act metabotropically, so they are best modelled as **changes to the
neuron's intrinsic parameters**, not as extra input.

## Part 2 — Noradrenaline as gain: scaling the f–I curve

A neuron's **f–I curve** plots its firing rate $f$ against input strength $I$. Its
**slope** is the neuron's *gain*: how many extra spikes per second you get for a bit more
input.

Noradrenaline is the textbook example of a **gain** signal. A simple way to represent this
in a model is a multiplicative **input gain** $g$ — the lever called `I_syn_gain` in the
table above — applied to the drive a neuron receives:

$$I_{\text{eff}}(t) = g \cdot I(t).$$

Raising $g$ steepens the f–I curve (more output per unit input) and lowers the rheobase
(the neuron reaches firing for weaker inputs). At the **network level** this rescales the
responsiveness of an entire population at once — exactly the kind of broadcast effect the
locus-coeruleus NA system is thought to provide.

Below: a **single-neuron + input-strength sweep**. The faint grey curve is the baseline
($g=1$); the blue curve is the NA-modulated neuron.

In [3]:
AMPS = np.linspace(0.0, 6.0, 30)   # input-strength sweep

def gain_modulation(g_NA=1.8, tau_m=20.0, probe_input=3.0):
    f_base = fi_curve(AMPS, tau_m, V_th0, gain=1.0)
    f_mod  = fi_curve(AMPS, tau_m, V_th0, gain=g_NA)

    r_base = firing_rate(probe_input, tau_m, V_th0, gain=1.0)
    r_mod  = firing_rate(probe_input, tau_m, V_th0, gain=g_NA)

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(AMPS, f_base, color=GREY, label="baseline gain $g=1$")
    ax.plot(AMPS, f_mod, color=BLUE, label=f"NA gain $g={g_NA:.1f}$")

    ax.axvline(probe_input, color="k", ls=":", alpha=0.5)
    ax.plot(probe_input, r_base, "o", color=GREY)
    ax.plot(probe_input, r_mod, "o", color=BLUE)
    ax.annotate(f"+{r_mod - r_base:.0f} Hz",
                xy=(probe_input, (r_base + r_mod) / 2),
                xytext=(probe_input + 0.3, (r_base + r_mod) / 2),
                fontsize=11, color=BLUE)

    ax.set_xlabel("input amplitude  $I$")
    ax.set_ylabel("firing rate  $f$  (Hz)")
    ax.set_title("NA as gain: a steeper f\u2013I curve", fontweight="bold")
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout(); plt.show()


interact(
    gain_modulation,
    g_NA=FloatSlider(value=1.8, min=1.0, max=3.0, step=0.1,
                     description="NA gain g", continuous_update=False),
    tau_m=FloatSlider(value=20.0, min=5.0, max=60.0, step=1.0,
                      description="\u03c4_m", continuous_update=False),
    probe_input=FloatSlider(value=3.0, min=1.0, max=6.0, step=0.1,
                            description="probe I", continuous_update=False),
);


interactive(children=(FloatSlider(value=1.8, continuous_update=False, description='NA gain g', max=3.0, min=1.…

## Part 3 — From gain to intrinsics: NA's metabotropic levers

Part 2 *described* NA as a gain knob. But where does that gain physically come from? NA
acts largely through **metabotropic adrenergic receptors**, whose intracellular cascades
modulate intrinsic conductances. One of the most important targets is **spike-frequency
adaptation** — the slow build-up of an adaptation current that makes a neuron fire more
slowly the longer it is driven. NA (and ACh) reduce these adaptation currents, which
*steepens the f–I slope* — i.e. produces gain.

So "NA = gain" and "NA = metabotropic modulation of intrinsics" are two descriptions of
the same thing. Here a single **NA-drive** slider simultaneously moves three biophysical
levers, the same ones used in many neuromorphic neuromodulation schemes:

- **input gain** $g$ — rises with NA,
- **adaptation strength** $b$ — falls with NA (less spike-frequency adaptation),
- **threshold** $V_{th}$ — lowers slightly with NA (more excitable).

Watch the f–I curve steepen (left) *and* the within-train adaptation disappear (right) as
you raise NA.

In [4]:
def na_levers(na):
    """Map a single NA-drive value in [0, 1] onto three intrinsic levers."""
    g   = 1.0 + 1.0 * na          # input gain      : 1.0 -> 2.0
    b   = 1.2 * (1.0 - 0.85 * na) # adaptation      : strong -> weak
    vth = V_th0 - 3.0 * na        # threshold (mV)  : -50 -> -53
    return g, b, vth


def na_drive(NA=0.5, probe_input=4.0):
    g, b, vth = na_levers(NA)
    g0, b0, vth0 = na_levers(0.0)

    f_base = fi_curve(AMPS, tau0, vth0, gain=g0, adapt_b=b0)
    f_mod  = fi_curve(AMPS, tau0, vth,  gain=g,  adapt_b=b)

    # within-train adaptation at a fixed probe input
    tt = np.arange(0.0, 400.0, 0.1)
    I = np.full_like(tt, probe_input)
    _, _, s0 = simulate_alif(tt, g0 * I, tau0, vth0, b=b0, dt_=0.1)
    _, _, s1 = simulate_alif(tt, g * I,  tau0, vth,  b=b,  dt_=0.1)

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    ax[0].plot(AMPS, f_base, color=GREY, label="NA = 0")
    ax[0].plot(AMPS, f_mod, color=BLUE, label=f"NA = {NA:.2f}")
    ax[0].set_xlabel("input amplitude $I$"); ax[0].set_ylabel("firing rate (Hz)")
    ax[0].set_title("f\u2013I curve", fontweight="bold")
    ax[0].legend(frameon=False, loc="upper left")

    for s, c, lab in [(s0, GREY, "NA = 0"), (s1, BLUE, f"NA = {NA:.2f}")]:
        if len(s) > 1:
            ax[1].plot(s[1:], 1000.0 / np.diff(s), "-o", ms=4, color=c,
                       label=f"{lab}  ({len(s)} spikes)")
    ax[1].set_xlabel("time (ms)"); ax[1].set_ylabel("instantaneous rate  1/ISI (Hz)")
    ax[1].set_title(f"spike-frequency adaptation (probe I = {probe_input:.1f})",
                    fontweight="bold")
    ax[1].legend(frameon=False)

    fig.suptitle(f"NA drive = {NA:.2f}   \u2192   "
                 f"gain g={g:.2f}, adaptation b={b:.2f}, $V_{{th}}$={vth:.1f} mV",
                 fontweight="bold")
    fig.tight_layout(); plt.show()


interact(
    na_drive,
    NA=FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                   description="NA drive", continuous_update=False),
    probe_input=FloatSlider(value=4.0, min=2.0, max=6.0, step=0.2,
                            description="probe I", continuous_update=False),
);


interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='NA drive', max=1.0, step=0.…

### 💬 Discussion — based on the notebook

1. **Part 1.** You can reach the same spike count by scaling the input (ionotropic) or by
   lowering $V_{th}$ (metabotropic). If you could only *measure* the neuron's voltage and
   spikes, could you tell the two mechanisms apart? What extra measurement would help?
2. **Part 2.** Increasing the gain $g$ both steepens the f–I curve *and* lowers the
   rheobase. Is that still "pure" multiplicative gain? What would a *purely* multiplicative
   gain change look like, and what biophysical ingredient (hint: background noise / balanced
   input) is usually needed to get it?
3. **Part 3.** The single NA-drive slider moves gain, adaptation and threshold together.
   Turn them one at a time (edit `na_levers`): which lever contributes most to the change in
   f–I slope? Which mainly shifts the curve sideways?
4. **Towards hardware.** On neuromorphic hardware you often expose exactly these knobs —
   $\tau_m$, $V_{th}$, adaptation — per neuron. Why is implementing neuromodulation as a
   change to *intrinsic parameters* (rather than to synaptic weights) attractive for
   on-chip continual learning?


## Optional — single-neuron explorer

A minimal panel to inspect one LIF neuron at a time by directly setting $\tau_m$, $V_{th}$,
the input amplitude and the pulse width.

In [5]:
def plot_single_lif(tau_m=20.0, v_th=-50.0, input_amp=2.2, input_width=75):
    I = make_rect_pulse(t, pulse_start, input_width, input_amp)
    V, spikes = simulate_lif(t, I, tau_m, v_th)

    fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                           gridspec_kw={"height_ratios": [1, 2]})
    ax[0].plot(t, I, color="slategray"); ax[0].set_ylabel("input")
    ax[1].axhline(v_th, color="gray", ls="--", label="threshold")
    ax[1].plot(t, V, color=BLUE)
    ax[1].set_ylabel("voltage (mV)"); ax[1].set_xlabel("time (ms)")
    ax[1].legend(frameon=False)
    fig.suptitle(f"\u03c4_m = {tau_m:.0f} ms | V_th = {v_th:.0f} mV | "
                 f"spikes = {len(spikes)}", fontweight="bold")
    fig.tight_layout(); plt.show()


interact(
    plot_single_lif,
    tau_m=FloatSlider(value=20.0, min=5.0, max=60.0, step=1.0,
                      description="\u03c4_m", continuous_update=False),
    v_th=FloatSlider(value=-50.0, min=-60.0, max=-40.0, step=1.0,
                     description="V_th", continuous_update=False),
    input_amp=FloatSlider(value=2.2, min=0.0, max=6.0, step=0.1,
                          description="input amp", continuous_update=False),
    input_width=IntSlider(value=75, min=5, max=100, step=5,
                          description="input width", continuous_update=False),
);


interactive(children=(FloatSlider(value=20.0, continuous_update=False, description='τ_m', max=60.0, min=5.0, s…

---

## Citation

This tutorial accompanies the perspective described in:

**Rodriguez-Garcia, A., Ghosh, A., Mei, J., & Ramaswamy, S. (2025).**
*Augmenting learning in neuro-embodied systems through neurobiological first principles.*
arXiv:2407.04525. <https://arxiv.org/abs/2407.04525>

```bibtex
@misc{rodriguezgarcia2025augmentinglearningneuroembodiedsystems,
      title={Augmenting learning in neuro-embodied systems through neurobiological first principles},
      author={Alejandro Rodriguez-Garcia and Anindya Ghosh and Jie Mei and Srikanth Ramaswamy},
      year={2025},
      eprint={2407.04525},
      archivePrefix={arXiv},
      primaryClass={q-bio.NC},
      url={https://arxiv.org/abs/2407.04525},
}
```

*Neural Circuits Group — Newcastle University | Contact: <Srikanth.Ramaswamy@newcastle.ac.uk>*
